In [ ]:
!python -m pip install -e ..

In [4]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from torch_measure.datasets import load
from torch_measure.models import Rasch, TwoPL, ThreePL, BetaRasch, BetaTwoPL
from torch_measure.data import ResponseMatrix
from torch_measure.viz import plot_icc, plot_response_heatmap, plot_information
import torch_measure
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

import warnings
from functools import partial
warnings.filterwarnings("ignore")

Using device: cpu


### Continuous-score note

This copy keeps the same IRT-style fitting flow, but the observations are soft HarmJudge prompt scores rather than binary correctness labels. For regular 1PL/2PL fits, use Torch Measure's `BetaRasch` and `BetaTwoPL`, which fit continuous responses in `(0, 1)` with a Beta likelihood. The 3PL and item-marginal custom fits remain exploratory soft-label logistic fits because Torch Measure does not currently expose a `BetaThreePL` or beta item-marginal fitter.


### Load the continuous HarmJudge safety-judge response matrix

Rows are HarmJudge models, columns are the 117 regular safety-judge prompts, and values are `overall_effectiveness_score` in `[0, 1]`. Treat these as soft/fractional responses rather than classical binary IRT observations. Missing prompt scores stay as `NaN` and are masked by the fitting code.


In [5]:
matrix_path = "../benchmarks/safety/final_judge_results/response_matrices/harmjudge_safety_solver_response_matrix.csv"
subject_meta_path = "../benchmarks/safety/final_judge_results/response_matrices/harmjudge_safety_solver_subject_metadata.csv"
item_meta_path = "../benchmarks/safety/final_judge_results/response_matrices/harmjudge_safety_solver_item_metadata.csv"

df = pd.read_csv(matrix_path, index_col=0)
subject_meta = pd.read_csv(subject_meta_path)
item_meta = pd.read_csv(item_meta_path)

rm = ResponseMatrix(
    data=torch.tensor(df.values, dtype=torch.float32),
    subject_ids=list(df.index.astype(str)),
    item_ids=list(df.columns.astype(str)),
    item_contents=list(item_meta["source"].astype(str)),
    subject_metadata=subject_meta.to_dict("records"),
    info={
        "benchmark_id": "harmjudge_safety_judge",
        "item_id_field": "prompt_id",
        "value": "overall_effectiveness_score: soft/fractional score in [0, 1]",
    },
)

observed_values = rm.data[~torch.isnan(rm.data)]

# Beta IRT models require strictly interior values, so keep NaNs but clip
# observed 0/1 scores into (eps, 1 - eps) for likelihood fitting.
BETA_EPS = 1e-4
BETA_PHI = 10.0
rm_beta_data = rm.data.clone()
rm_beta_mask = ~torch.isnan(rm_beta_data) & (rm_beta_data != -1)
rm_beta_data[rm_beta_mask] = rm_beta_data[rm_beta_mask].clamp(BETA_EPS, 1 - BETA_EPS)

print(f"{rm.n_subjects} models x {rm.n_items} items, density = {rm.density:.1%}")
print(f"value range: {observed_values.min().item():.4f} to {observed_values.max().item():.4f}")
print(f"Beta-fit clipping: [{BETA_EPS}, {1 - BETA_EPS}], phi={BETA_PHI}")


8 models x 117 items, density = 99.3%
value range: 0.0000 to 1.0000
Beta-fit clipping: [0.0001, 0.9999], phi=10.0


### Regular IRT fits

These fits keep the usual orientation: models are subjects and safety-judge prompts are items. For continuous scores, `1pl_regular` uses `BetaRasch` and `2pl_regular` uses `BetaTwoPL`; `3pl_regular` is kept as a soft-label Bernoulli 3PL comparison because there is no beta 3PL implementation in Torch Measure.


In [6]:
def center_item_difficulty(model):
    with torch.no_grad():
        shift = model.difficulty.mean()
        model.difficulty.sub_(shift)
        model.ability.sub_(shift)
    return model


regular_fits = {}


def soft_bernoulli_nll(predicted_probs, observed):
    predicted_probs = predicted_probs.clamp(1e-7, 1 - 1e-7)
    return -(
        observed * torch.log(predicted_probs)
        + (1 - observed) * torch.log1p(-predicted_probs)
    ).mean()

# Spec format: (model_factory, fit_kwargs, fit_data).
# BetaRasch/BetaTwoPL use the clipped continuous matrix; 3PL stays on the
# original soft scores as an exploratory Bernoulli-style comparison.
regular_specs = {
    "1pl_regular": (partial(BetaRasch, phi=BETA_PHI), {"max_epochs": 10000, "lr": 0.01}, rm_beta_data),
    "2pl_regular": (partial(BetaTwoPL, phi=BETA_PHI), {"max_epochs": 10000, "lr": 0.005}, rm_beta_data),
    "3pl_regular": (ThreePL, {"max_epochs": 10000, "lr": 0.003, "loss_fn": soft_bernoulli_nll}, rm.data),
}

for fit_name, (model_factory, fit_kwargs, fit_data) in regular_specs.items():
    print(f"\nFitting {fit_name}")
    model = model_factory(rm.n_subjects, rm.n_items, device=device)
    history = model.fit(
        fit_data,
        method="mle",
        verbose=True,
        **fit_kwargs,
    )
    center_item_difficulty(model)
    regular_fits[fit_name] = {"model": model, "history": history}
    print(f"{fit_name} final loss: {history['losses'][-1]:.4f}")



Fitting 1pl_regular


MLE fitting:  27%|██▋       | 2736/10000 [00:06<00:18, 402.47it/s, loss=3.355638]


1pl_regular final loss: 3.3556

Fitting 2pl_regular


MLE fitting:  32%|███▏      | 3244/10000 [00:10<00:22, 301.25it/s, loss=2.862693]


2pl_regular final loss: 2.8627

Fitting 3pl_regular


MLE fitting:  27%|██▋       | 2702/10000 [00:08<00:23, 308.71it/s, loss=0.309176]

3pl_regular final loss: 0.3092


### 1PL item-marginal MMLE

The library's built-in EM marginalizes over abilities. For model capability, this custom 1PL fit instead integrates out item difficulties and estimates only model abilities.

In [ ]:
rasch_item_marginal = Rasch(
    n_subjects=rm.n_items,      # original items/tasks
    n_items=rm.n_subjects,      # original models
    device=device,
)

hist_1pl_item_marginal = rasch_item_marginal.fit(
    rm.data.T,
    method="em",
    max_epochs=500,
    lr=0.01,
    n_quadrature=31,
    latent_prior_sd =1.5,
    verbose=True,
)

# After flipping, each original model is an item. Lower fitted difficulty means
# stronger model capability, so capability is negative difficulty.
theta_1pl_item_marginal = -rasch_item_marginal.difficulty.detach().cpu()
theta_1pl_item_marginal = theta_1pl_item_marginal - theta_1pl_item_marginal.mean()

print(f"1PL item-marginal final loss: {hist_1pl_item_marginal['losses_ability'][-1]:.4f}")

### 2PL and 3PL item-marginalized IRT

In [ ]:
import math
import torch.nn.functional as F
from tqdm.auto import tqdm

# a, c <-- random or still fitting?

def fit_abilities_item_marginalized_pl(
    data,
    pl=2,
    n_samples=1024,
    max_epochs=1000,
    lr=0.03,
    b_sd=1.5,
    log_a_mean=0.0,
    log_a_sd=0.5,
    c_alpha=1.0,
    c_beta=9.0,
    ability_prior_sd=3.0,
    device="cpu",
    seed=0,
):
    assert pl in {2, 3}

    Y = data.to(device).float()
    mask = ~torch.isnan(Y) & (Y != -1)

    n_models, n_items = Y.shape
    ability = torch.nn.Parameter(torch.zeros(n_models, device=device))

    gen = torch.Generator(device=device)
    gen.manual_seed(seed)

    b_samples = torch.randn(n_samples, generator=gen, device=device) * b_sd

    log_a_samples = (
        torch.randn(n_samples, generator=gen, device=device) * log_a_sd
        + log_a_mean
    )
    a_samples = torch.exp(log_a_samples)

    if pl == 3:
        beta = torch.distributions.Beta(
            torch.tensor(c_alpha, device=device),
            torch.tensor(c_beta, device=device),
        )
        c_samples = beta.sample((n_samples,))
    else:
        c_samples = torch.zeros(n_samples, device=device)

    log_weight = -math.log(n_samples)
    n_obs = mask.sum().clamp_min(1).float()

    opt = torch.optim.Adam([ability], lr=lr)
    losses = []

    for _ in tqdm(range(max_epochs), desc=f"{pl}PL item-marginal MMLE"):
        opt.zero_grad()
        item_terms = []

        for item_idx in range(n_items):
            obs = mask[:, item_idx]
            if not obs.any():
                continue

            y = Y[obs, item_idx]
            theta = ability[obs]

            logits = a_samples[None, :] * (
                theta[:, None] - b_samples[None, :]
            )

            if pl == 2:
                logp = (
                    y[:, None] * F.logsigmoid(logits)
                    + (1 - y[:, None]) * F.logsigmoid(-logits)
                )
            else:
                p = c_samples[None, :] + (1 - c_samples[None, :]) * torch.sigmoid(logits)
                p = p.clamp(1e-7, 1 - 1e-7)
                logp = y[:, None] * torch.log(p) + (1 - y[:, None]) * torch.log1p(-p)

            item_terms.append(torch.logsumexp(log_weight + logp.sum(dim=0), dim=0))

        log_likelihood = torch.stack(item_terms).sum()
        prior = 0.5 * (ability / ability_prior_sd).pow(2).mean()
        loss = -log_likelihood / n_obs + prior

        loss.backward()
        opt.step()
        losses.append(loss.item())

    return ability.detach(), losses

In [ ]:
theta_2pl_item_marginal, loss_2pl_item_marginal = fit_abilities_item_marginalized_pl(
    rm.data,
    pl=2,
    n_samples=2048,
    max_epochs=1000,
    lr=0.03,
    b_sd=1.5,
    device=device,
    seed=0,
)

theta_2pl_item_marginal = theta_2pl_item_marginal - theta_2pl_item_marginal.mean()

In [ ]:
theta_3pl_item_marginal, loss_3pl_item_marginal = fit_abilities_item_marginalized_pl(
    rm.data,
    pl=3,
    n_samples=4096,
    max_epochs=1000,
    lr=0.03,
    b_sd=1.5,
    c_alpha=1.0,
    c_beta=9.0,
    device=device,
    seed=0,
)

theta_3pl_item_marginal = theta_3pl_item_marginal - theta_3pl_item_marginal.mean()

### Final capability table

Mean score is included as a sanity-check reference. The final table includes both regular and item-marginalized capability estimates fit to soft/fractional responses.

In [9]:
mean_score = df.mean(axis=1, skipna=True)
n_observed = df.notna().sum(axis=1)

capability_df = pd.DataFrame({
    "model": rm.subject_ids,
    "mean_score": mean_score.loc[rm.subject_ids].values,
    "n_observed": n_observed.loc[rm.subject_ids].values,
    "1pl_regular": regular_fits["1pl_regular"]["model"].ability.detach().cpu().numpy(),
    # "1pl_item_marginal_mmle": theta_1pl_item_marginal.cpu().numpy(),
    "2pl_regular": regular_fits["2pl_regular"]["model"].ability.detach().cpu().numpy(),
    # "2pl_item_marginal_mmle": theta_2pl_item_marginal.cpu().numpy(),
    "3pl_regular": regular_fits["3pl_regular"]["model"].ability.detach().cpu().numpy(),
    # "3pl_item_marginal_mmle": theta_3pl_item_marginal.cpu().numpy(),
})

ability_cols = [
    "1pl_regular",
    # "1pl_item_marginal_mmle",
    "2pl_regular",
    # "2pl_item_marginal_mmle",
    "3pl_regular",
    # "3pl_item_marginal_mmle"
]

for col in ability_cols:
    capability_df[f"{col}_rank"] = capability_df[col].rank(
        ascending=False,
        method="min",
    ).astype(int)

capability_df = capability_df.sort_values("1pl_regular", ascending=False)
capability_df

,model,mean_score,n_observed,1pl_regular,2pl_regular,3pl_regular,1pl_regular_rank,2pl_regular_rank,3pl_regular_rank
4,qwen3_5_27b,0.784188,117,4.455584,0.469483,0.330542,1,1,1
0,claude_sonnet_4_6,0.679545,110,2.843342,-0.296668,0.084414,2,4,2
1,ministral_3_14b_instruct_2512_bf16,0.557692,117,0.270317,-0.289095,-0.153348,3,2,3
3,ministral_3_8b_instruct_2512_bf16,0.574786,117,-0.034734,-0.289301,-0.181834,4,3,4
6,qwen3_5_4b,0.478632,117,-2.034943,-0.756475,-0.223181,5,5,5
7,qwen3_5_9b,0.192308,117,-5.057278,-1.249110,-0.689127,6,6,6
2,ministral_3_3b_instruct_2512_bf16,0.098291,117,-6.302899,-1.492721,-2.362085,7,7,8
5,qwen3_5_2b,0.027778,117,-7.170772,-1.646992,-2.136863,8,8,7


### Uncertainty quantification

For the regular IRT fits, use a diagonal Laplace/Fisher approximation for ability standard errors. For item-marginal fits, bootstrap over item columns because the item parameters have been integrated out.

In [ ]:
def ability_laplace_se(model, data):
    data = data.to(model.device).float()
    mask = ~torch.isnan(data) & (data != -1)

    with torch.no_grad():
        theta = model.ability.detach()
        beta = model.difficulty.detach()

        if hasattr(model, "discrimination"):
            a = model.discrimination.detach()
        else:
            a = torch.ones_like(beta)

        logits = a.unsqueeze(0) * (theta.unsqueeze(1) - beta.unsqueeze(0))
        base_prob = torch.sigmoid(logits)

        if hasattr(model, "guessing"):
            c = model.guessing.detach()
            prob = c.unsqueeze(0) + (1 - c.unsqueeze(0)) * base_prob
            dprob_dtheta = (1 - c.unsqueeze(0)) * a.unsqueeze(0) * base_prob * (1 - base_prob)
        else:
            prob = base_prob
            dprob_dtheta = a.unsqueeze(0) * base_prob * (1 - base_prob)

        prob = prob.clamp(1e-7, 1 - 1e-7)
        info_matrix = dprob_dtheta.pow(2) / (prob * (1 - prob))
        info_matrix = torch.where(mask, info_matrix, torch.zeros_like(info_matrix))
        info = info_matrix.sum(dim=1)
        se = 1 / torch.sqrt(info.clamp_min(1e-8))

    return se.detach().cpu()


for fit_name in ["1pl_regular", "2pl_regular", "3pl_regular"]:
    se = ability_laplace_se(regular_fits[fit_name]["model"], rm.data)
    capability_df[f"{fit_name}_se"] = se.numpy()
    capability_df[f"{fit_name}_lo"] = capability_df[fit_name] - 1.96 * capability_df[f"{fit_name}_se"]
    capability_df[f"{fit_name}_hi"] = capability_df[fit_name] + 1.96 * capability_df[f"{fit_name}_se"]

capability_df[
    ["model", "mean_score", "n_observed"]
    + [
        "1pl_regular", "1pl_regular_se", "1pl_regular_lo", "1pl_regular_hi",
        "2pl_regular", "2pl_regular_se", "2pl_regular_lo", "2pl_regular_hi",
        "3pl_regular", "3pl_regular_se", "3pl_regular_lo", "3pl_regular_hi",
    ]
]

In [ ]:
def fit_1pl_item_marginal_capability(data, max_epochs=200, lr=0.01, n_quadrature=31, device="cpu", seed=0):
    model = Rasch(
        n_subjects=data.shape[1],
        n_items=data.shape[0],
        device=device,
    )
    _ = model.fit(
        data.T,
        method="em",
        max_epochs=max_epochs,
        lr=lr,
        n_quadrature=n_quadrature,
        verbose=False,
    )
    theta = -model.difficulty.detach().cpu()
    return theta - theta.mean()


def bootstrap_item_capabilities(data, fit_fn, fit_kwargs, n_boot=50, seed=0):
    rng = np.random.default_rng(seed)
    n_models, n_items = data.shape
    boot = []

    for boot_idx in tqdm(range(n_boot), desc="Bootstrap items"):
        cols = rng.integers(0, n_items, size=n_items)
        data_b = data[:, cols]
        theta_b = fit_fn(data_b, **fit_kwargs, seed=seed + boot_idx)
        theta_b = theta_b.detach().cpu()
        theta_b = theta_b - theta_b.mean()
        boot.append(theta_b.numpy())

    return np.vstack(boot)


def add_bootstrap_summary(capability_df, boot, subject_ids, col):
    summary = pd.DataFrame({
        "model": subject_ids,
        f"{col}_boot_mean": boot.mean(axis=0),
        f"{col}_boot_se": boot.std(axis=0, ddof=1),
        f"{col}_boot_lo": np.percentile(boot, 2.5, axis=0),
        f"{col}_boot_hi": np.percentile(boot, 97.5, axis=0),
    })
    return capability_df.merge(summary, on="model", how="left")


# Set to True when you want bootstrap SEs for item-marginal capabilities.
RUN_ITEM_BOOTSTRAP = False
N_BOOT = 50

if RUN_ITEM_BOOTSTRAP:
    boot_1pl = bootstrap_item_capabilities(
        rm.data,
        fit_1pl_item_marginal_capability,
        fit_kwargs={"max_epochs": 200, "lr": 0.01, "n_quadrature": 31, "device": device},
        n_boot=N_BOOT,
        seed=101,
    )
    capability_df = add_bootstrap_summary(capability_df, boot_1pl, rm.subject_ids, "1pl_item_marginal_mmle")

    boot_2pl = bootstrap_item_capabilities(
        rm.data,
        fit_abilities_item_marginalized_pl,
        fit_kwargs={"pl": 2, "n_samples": 1024, "max_epochs": 300, "lr": 0.03, "b_sd": 1.5, "device": device},
        n_boot=N_BOOT,
        seed=202,
    )
    capability_df = add_bootstrap_summary(capability_df, boot_2pl, rm.subject_ids, "2pl_item_marginal_mmle")

    boot_3pl = bootstrap_item_capabilities(
        rm.data,
        fit_abilities_item_marginalized_pl,
        fit_kwargs={"pl": 3, "n_samples": 2048, "max_epochs": 300, "lr": 0.03, "b_sd": 1.5, "c_alpha": 1.0, "c_beta": 9.0, "device": device},
        n_boot=N_BOOT,
        seed=303,
    )
    capability_df = add_bootstrap_summary(capability_df, boot_3pl, rm.subject_ids, "3pl_item_marginal_mmle")

capability_df

### Heldout Evaluation - Model Selection

Use an 80:20 split over observed cells. Each candidate model is refit on the training cells and predicts held-out soft scores. `BetaRasch`/`BetaTwoPL` fit on the clipped beta matrix but evaluate predictions against the original continuous scores. Treat Brier/ECE as more interpretable than AUC for continuous targets; AUC assumes binary labels.


In [ ]:
from torch_measure.data import random_mask
from torch_measure.metrics.functional import compute_all
from torch_measure.models import predict_dense


def _heldout_metric_row(fit_name, rep, metrics, data, test_mask):
    y_test = data[test_mask].float()
    return {
        "fit": fit_name,
        "rep": rep,
        "n_test": int(test_mask.sum().item()),
        "test_positive_rate": y_test.mean().item(),
        "heldout_auc": metrics["auc"],
        "heldout_log_likelihood": metrics["log_likelihood"],
        "heldout_brier": metrics["brier"],
        "heldout_ece": metrics["ece"],
    }


def heldout_auc_for_regular_model(model_factory, fit_data, fit_kwargs, train_mask, test_mask, eval_data=None):
    if eval_data is None:
        eval_data = fit_data

    model = model_factory(fit_data.shape[0], fit_data.shape[1], device=device)
    history = model.fit(
        fit_data,
        mask=train_mask,
        method="mle",
        verbose=False,
        **fit_kwargs,
    )
    center_item_difficulty(model)

    probs = predict_dense(model).detach()
    metrics = compute_all(probs, eval_data, mask=test_mask)
    return metrics, model, history


def heldout_auc_for_1pl_item_marginal(data, fit_kwargs, train_mask, test_mask):
    flipped = Rasch(
        n_subjects=data.shape[1],
        n_items=data.shape[0],
        device=device,
    )
    history = flipped.fit(
        data.T,
        mask=train_mask.T,
        method="em",
        verbose=False,
        **fit_kwargs,
    )
    center_item_difficulty(flipped)

    probs = predict_dense(flipped).detach().T
    metrics = compute_all(probs, data, mask=test_mask)
    return metrics, flipped, history


def item_marginal_prior_probs(
    theta,
    n_items,
    pl,
    n_samples=8192,
    b_sd=1.5,
    log_a_mean=0.0,
    log_a_sd=0.5,
    c_alpha=1.0,
    c_beta=9.0,
    seed=0,
):
    theta = theta.to(device).float()
    torch.manual_seed(seed)
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)

    b_samples = torch.randn(n_samples, generator=gen, device=device) * b_sd
    log_a_samples = torch.randn(n_samples, generator=gen, device=device) * log_a_sd + log_a_mean
    a_samples = torch.exp(log_a_samples)

    logits = a_samples[None, :] * (theta[:, None] - b_samples[None, :])
    if pl == 2:
        p_samples = torch.sigmoid(logits)
    elif pl == 3:
        beta = torch.distributions.Beta(
            torch.tensor(c_alpha, device=device),
            torch.tensor(c_beta, device=device),
        )
        c_samples = beta.sample((n_samples,))
        p_samples = c_samples[None, :] + (1 - c_samples[None, :]) * torch.sigmoid(logits)
    else:
        raise ValueError(f"Expected pl=2 or pl=3, got {pl!r}")

    p_model = p_samples.mean(dim=1)
    return p_model[:, None].expand(-1, n_items)


def heldout_auc_for_prior_item_marginal(data, fit_kwargs, predict_kwargs, train_mask, test_mask, seed=0):
    train_data = data.clone()
    train_data[~train_mask] = float("nan")

    theta, history = fit_abilities_item_marginalized_pl(
        train_data,
        **fit_kwargs,
        seed=seed,
    )
    theta = theta - theta.mean()

    probs = item_marginal_prior_probs(
        theta,
        n_items=data.shape[1],
        seed=seed + 100_000,
        **predict_kwargs,
    ).detach()
    metrics = compute_all(probs, data, mask=test_mask)
    return metrics, theta, history


item_marginal_specs = {
    "1pl_item_marginal_mmle": {
        "kind": "flipped_1pl",
        "fit_kwargs": {"max_epochs": 500, "lr": 0.01, "n_quadrature": 31},
    },
    "2pl_item_marginal_mmle": {
        "kind": "prior_marginal",
        "fit_kwargs": {"pl": 2, "n_samples": 2048, "max_epochs": 1000, "lr": 0.03, "b_sd": 1.5, "device": device},
        "predict_kwargs": {"pl": 2, "n_samples": 8192, "b_sd": 1.5},
    },
    "3pl_item_marginal_mmle": {
        "kind": "prior_marginal",
        "fit_kwargs": {"pl": 3, "n_samples": 4096, "max_epochs": 1000, "lr": 0.03, "b_sd": 1.5, "c_alpha": 1.0, "c_beta": 9.0, "device": device},
        "predict_kwargs": {"pl": 3, "n_samples": 8192, "b_sd": 1.5, "c_alpha": 1.0, "c_beta": 9.0},
    },
}


def evaluate_heldout_auc(regular_specs, item_marginal_specs, data, n_repeats=1, train_frac=0.8, seed=0):
    data = data.to(device).float()
    observed = ~torch.isnan(data) & (data != -1)
    rows = []

    for rep in range(n_repeats):
        torch.manual_seed(seed + rep)
        train_mask, test_mask = random_mask(observed, train_frac=train_frac)

        for fit_name, spec in regular_specs.items():
            if len(spec) == 2:
                model_factory, fit_kwargs = spec
                fit_data = data
            else:
                model_factory, fit_kwargs, fit_data = spec
                fit_data = fit_data.to(device).float()

            metrics, _, _ = heldout_auc_for_regular_model(
                model_factory,
                fit_data,
                fit_kwargs,
                train_mask=train_mask,
                test_mask=test_mask,
                eval_data=data,
            )
            rows.append(_heldout_metric_row(fit_name, rep, metrics, data, test_mask))

        for fit_name, spec in item_marginal_specs.items():
            if spec["kind"] == "flipped_1pl":
                metrics, _, _ = heldout_auc_for_1pl_item_marginal(
                    data,
                    spec["fit_kwargs"],
                    train_mask=train_mask,
                    test_mask=test_mask,
                )
            elif spec["kind"] == "prior_marginal":
                metrics, _, _ = heldout_auc_for_prior_item_marginal(
                    data,
                    spec["fit_kwargs"],
                    spec["predict_kwargs"],
                    train_mask=train_mask,
                    test_mask=test_mask,
                    seed=seed + rep,
                )
            else:
                raise ValueError(f"Unknown item-marginal spec kind: {spec['kind']!r}")

            rows.append(_heldout_metric_row(fit_name, rep, metrics, data, test_mask))

    return pd.DataFrame(rows)

In [ ]:
heldout_eval_raw = evaluate_heldout_auc(
    regular_specs,
    item_marginal_specs,
    rm.data,
    n_repeats=1,
    train_frac=0.8,
    seed=123,
)

heldout_eval_summary = (
    heldout_eval_raw
    .groupby("fit")
    .agg(
        n_test_mean=("n_test", "mean"),
        test_positive_rate_mean=("test_positive_rate", "mean"),
        heldout_auc_mean=("heldout_auc", "mean"),
        heldout_auc_sd=("heldout_auc", "std"),
        heldout_log_likelihood_mean=("heldout_log_likelihood", "mean"),
        heldout_log_likelihood_sd=("heldout_log_likelihood", "std"),
        heldout_brier_mean=("heldout_brier", "mean"),
        heldout_brier_sd=("heldout_brier", "std"),
        heldout_ece_mean=("heldout_ece", "mean"),
        heldout_ece_sd=("heldout_ece", "std"),
    )
    .reset_index()
    .sort_values("heldout_auc_mean", ascending=False)
)

heldout_eval_summary